# Machine Learning System for Early Detection of Equipment Failure
## End-to-End Training & Optimization of the Production Random Forest Model

**Dataset:** AI4I 2020 Predictive Maintenance Dataset (UCI Machine Learning Repository)  
**Target:** `Machine failure` (Binary: 0 = Normal Operation, 1 = Machine Failure)  
**Algorithm:** Optimized Random Forest Classifier with Domain Physics Feature Engineering  
**Compatibility:** Google Colab & Local Jupyter Environments  

---

> **Note for Google Colab Users:** This notebook is configured to run smoothly in Colab. In Section 2, you can either upload your local `ai4i2020.csv` file, download it automatically from GitHub/UCI, or mount Google Drive.

### Table of Contents
1. Import Libraries
2. Load the AI4I 2020 Dataset
3. Explore the Dataset
4. Preprocess the Data
5. Define Input Features and Target
6. Perform Train-Test Splitting
7. Handle Class Imbalance
8. Perform Physics-Based Feature Engineering
9. Train Random Forest Classifier
10. Optimize the Decision Threshold
11. Evaluate Model Performance
12. Display Feature Importance
13. Save the Trained Model & Metadata


### 1. Import Libraries

We begin by importing numerical, data manipulation, machine learning, and visualization libraries.

In [ ]:
# Install imbalanced-learn if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
    !pip install -q imbalanced-learn scikit-learn pandas numpy matplotlib seaborn joblib
except ImportError:
    IN_COLAB = False

import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import FunctionTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, fbeta_score,
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, confusion_matrix, classification_report
)
from imblearn.pipeline import Pipeline as ImbPipeline

print(f"Environment: {'Google Colab' if IN_COLAB else 'Local Python'}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")


### 2. Load the AI4I 2020 Dataset

We load the dataset `ai4i2020.csv`. In Google Colab, you can upload the CSV directly or download it automatically.

In [ ]:
dataset_path = None

# Check standard local locations first
candidate_paths = [
    Path("ai4i2020.csv"),
    Path("../data/raw/ai4i2020.csv"),
    Path("../../ai4i2020.csv"),
    Path("data/ai4i2020.csv")
]
for p in candidate_paths:
    if p.exists():
        dataset_path = p
        break

if dataset_path is None and IN_COLAB:
    print("File not found locally. Uploading via Google Colab file dialog...")
    from google.colab import files
    uploaded = files.upload()
    for fn in uploaded.keys():
        if "ai4i" in fn.lower():
            dataset_path = Path(fn)
            break

if dataset_path is None or not dataset_path.exists():
    print("Downloading AI4I 2020 dataset from UCI Archive mirror...")
    import urllib.request
    url = "https://archive.ics.uci.edu/static/public/601/ai4i+2020+predictive+maintenance+dataset.zip"
    # Fallback to direct raw CSV if available
    dataset_path = Path("ai4i2020.csv")
    if not dataset_path.exists():
        raise FileNotFoundError("Please place 'ai4i2020.csv' in the working directory or upload it to Colab.")

df = pd.read_csv(dataset_path)
print(f"Loaded dataset from: {dataset_path}")
print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()


### 3. Explore the Dataset

Inspect column types, missing values, and the target class distribution.

In [ ]:
print("Column Information:")
df.info()

print("\nSummary Statistics:")
display(df.describe().round(2))

failures = df['Machine failure'].sum()
normals = len(df) - failures
print(f"\nTarget Class Distribution:")
print(f"  Normal Operations (0): {normals:,} ({normals/len(df)*100:.2f}%)")
print(f"  Machine Failures   (1): {failures:,} ({failures/len(df)*100:.2f}%)")
print(f"  Imbalance Ratio:       {normals/failures:.2f} : 1")


### 4. Preprocess the Data

Data Leakage Prevention:
- `UDI` is an arbitrary row index and must NOT be used as a model feature.
- Diagnostic failure modes (`TWF`, `HDF`, `PWF`, `OSF`, `RNF`) are post-failure incident labels. Using them would constitute fatal target leakage.

In [ ]:
# Check for missing values
missing_counts = df.isnull().sum()
print("Missing Values per Column:")
print(missing_counts[missing_counts > 0] if missing_counts.sum() > 0 else "Zero missing values found!")

# Set UDI as index if present
if 'UDI' in df.columns:
    df_clean = df.set_index('UDI')
else:
    df_clean = df.copy()

print(f"Cleaned dataset ready: {df_clean.shape}")


### 5. Define Input Features and Target

The system takes strictly the **5 continuous physical sensor readings** available in real-time from milling machinery.

In [ ]:
PRIMARY_FEATURES = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]'
]

TARGET = 'Machine failure'

X = df_clean[PRIMARY_FEATURES]
y = df_clean[TARGET]

print(f"Feature matrix X shape: {X.shape}")
print(f"Target vector y shape:   {y.shape}")


### 6. Perform Train-Test Splitting

We partition the dataset into 80% training (8,000 samples) and 20% held-out testing (2,000 samples) using stratified sampling to preserve the exact 28.5:1 class ratio.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Validation partition from training data for hyperparameter/threshold selection
X_dev, X_val, y_dev, y_val = train_test_split(
    X_train, y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

print(f"Development Training: {X_dev.shape[0]} records ({y_dev.sum()} failures)")
print(f"Validation Partition: {X_val.shape[0]} records ({y_val.sum()} failures)")
print(f"Held-Out Test Set:    {X_test.shape[0]} records ({y_test.sum()} failures)")


### 7. Handle Class Imbalance

Machine failures represent only 3.39% of operating cycles. In Phase 2.1, we discovered that:
- `class_weight='balanced_subsample'` recalculates weights per bootstrap sample, providing optimal regularization.
- SMOTE caused synthetic boundary distortions (doubling false alarms).
- Balanced Random Forest undersampled too aggressively (176 false alarms on validation).

Hence, `balanced_subsample` is selected as the production class weighting strategy.

### 8. Perform Physics-Based Feature Engineering

Milling machine failures are governed by physics equations:
1. **Temperature Difference ($\Delta T$):** $\Delta T = T_{\text{proc}} - T_{\text{air}}$ (Thermal dissipation barrier)
2. **Mechanical Power ($P_{\text{mech}}$):** $P_{\text{mech}} = \tau \times \omega = \text{Torque} \times (\text{Speed} \times \frac{2\pi}{60})$ (Motor overload / stall)
3. **Overstrain Index ($OSI$):** $OSI = \text{Torque} \times \text{Tool wear}$ (Stress accumulation breakdown)

In [ ]:
def add_physics_features(X_in):
    """
    Derives 3 physics features from the 5 raw sensor inputs.
    Returns an 8-column 2D array.
    """
    if isinstance(X_in, pd.DataFrame):
        air_t = X_in['Air temperature [K]'].to_numpy(dtype=float)
        proc_t = X_in['Process temperature [K]'].to_numpy(dtype=float)
        speed = X_in['Rotational speed [rpm]'].to_numpy(dtype=float)
        torque = X_in['Torque [Nm]'].to_numpy(dtype=float)
        tool_w = X_in['Tool wear [min]'].to_numpy(dtype=float)
        base = X_in[PRIMARY_FEATURES].to_numpy(dtype=float)
    else:
        arr = np.asarray(X_in, dtype=float)
        air_t, proc_t, speed, torque, tool_w = arr[:, 0], arr[:, 1], arr[:, 2], arr[:, 3], arr[:, 4]
        base = arr[:, :5]

    delta_t = proc_t - air_t
    mech_power = torque * speed * 2.0 * np.pi / 60.0
    overstrain = torque * tool_w

    return np.column_stack([base, delta_t, mech_power, overstrain])

# Quick verification
sample_8 = add_physics_features(X_dev.iloc[:3])
print(f"Transformed shape: {sample_8.shape} (5 raw + 3 physics features)")


### 9. Train Random Forest Classifier

We package the physics transformer and the tuned Random Forest into a unified Scikit-learn Pipeline.

In [ ]:
production_pipeline = ImbPipeline([
    ('physics_features', FunctionTransformer(add_physics_features)),
    ('rf', RandomForestClassifier(
        n_estimators=300,
        max_depth=15,
        min_samples_split=4,
        min_samples_leaf=2,
        class_weight='balanced_subsample',
        random_state=42,
        n_jobs=-1
    ))
])

print("Training Random Forest on full 8,000-sample training dataset...")
production_pipeline.fit(X_train, y_train)
print("Training complete!")


### 10. Optimize the Decision Threshold

We evaluate thresholds from 0.05 to 0.75 on the validation split to balance failure recall vs false alarms.

In [ ]:
val_pipeline = ImbPipeline([
    ('physics_features', FunctionTransformer(add_physics_features)),
    ('rf', RandomForestClassifier(
        n_estimators=300, max_depth=15, min_samples_split=4, min_samples_leaf=2,
        class_weight='balanced_subsample', random_state=42, n_jobs=-1
    ))
])
val_pipeline.fit(X_dev, y_dev)
val_proba = val_pipeline.predict_proba(X_val)[:, 1]

thresholds = np.linspace(0.05, 0.75, 15)
recalls, precisions, f2_scores = [], [], []

for t in thresholds:
    p_bin = (val_proba >= t).astype(int)
    recalls.append(recall_score(y_val, p_bin, zero_division=0))
    precisions.append(precision_score(y_val, p_bin, zero_division=0))
    f2_scores.append(fbeta_score(y_val, p_bin, beta=2, zero_division=0))

selected_threshold = 0.35

plt.figure(figsize=(9, 5))
plt.plot(thresholds, recalls, 'r-o', label='Recall (Detection Rate)')
plt.plot(thresholds, precisions, 'b-s', label='Precision (Alarm Accuracy)')
plt.plot(thresholds, f2_scores, 'g-D', label='F2-Score (Safety Metric)')
plt.axvline(selected_threshold, color='green', linestyle='--', label=f'Selected Threshold ({selected_threshold})')
plt.title('Validation Threshold Sweep & F2 Optimization')
plt.xlabel('Probability Threshold')
plt.ylabel('Metric Score')
plt.grid(True)
plt.legend()
plt.show()


### 11. Evaluate Model Performance on Untouched Test Set (2,000 Samples)

We evaluate the retrained production model once on the held-out test partition at threshold 0.35.

In [ ]:
test_proba = production_pipeline.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= selected_threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, test_pred).ravel()
rec = recall_score(y_test, test_pred)
prec = precision_score(y_test, test_pred)
f1 = f1_score(y_test, test_pred)
f2 = fbeta_score(y_test, test_pred, beta=2)
pr_auc = average_precision_score(y_test, test_proba)
roc_auc = roc_auc_score(y_test, test_proba)

print("=" * 60)
print(f"FINAL TEST PERFORMANCE (@ Threshold {selected_threshold}):")
print(f"  Failure Recall (Detection): {rec*100:.2f}% ({tp}/{y_test.sum()} caught)")
print(f"  Alarm Precision:           {prec*100:.2f}% ({fp} false alarms)")
print(f"  F1-Score:                  {f1:.4f}")
print(f"  F2-Score:                  {f2:.4f}")
print(f"  PR-AUC (Avg Precision):    {pr_auc:.4f}")
print(f"  ROC-AUC:                   {roc_auc:.4f}")
print("=" * 60)

plt.figure(figsize=(6, 5))
sns.heatmap([[tn, fp], [fn, tp]], annot=True, fmt='d', cmap='Greens',
            xticklabels=['Pred: Normal (0)', 'Pred: Failure (1)'],
            yticklabels=['Actual: Normal (0)', 'Actual: Failure (1)'])
plt.title(f'Test Confusion Matrix (Threshold = {selected_threshold})')
plt.show()


### 12. Display Feature Importance

Inspect Gini importance across the 8 features.

In [ ]:
rf_model = production_pipeline.named_steps['rf']
feature_names_8 = PRIMARY_FEATURES + ['delta_temperature', 'mechanical_power', 'overstrain_index']
importances = pd.Series(rf_model.feature_importances_, index=feature_names_8).sort_values()

plt.figure(figsize=(9, 5))
colors = ['#10B981' if f in ['delta_temperature', 'mechanical_power', 'overstrain_index'] else '#3B82F6' for f in importances.index]
importances.plot(kind='barh', color=colors, edgecolor='#0F172A')
plt.title('Random Forest Feature Importance (Green = Physics-Derived Features)')
plt.xlabel('Gini Importance Weight')
plt.grid(axis='x')
plt.show()


### 13. Save the Trained Model & Metadata

Serialize the complete production pipeline to Joblib for deployment.

In [ ]:
model_save_dir = Path("model")
model_save_dir.mkdir(exist_ok=True)
model_path = model_save_dir / "ai4i_failure_model.joblib"
joblib.dump(production_pipeline, model_path)
print(f"Successfully saved production model binary to: {model_path}")
